In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np

# Transforms
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomResizedCrop(64, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.3444, 0.3803, 0.4078],
                         std=[0.2038, 0.1367, 0.1147])
])

val_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.3444, 0.3803, 0.4078],
                         std=[0.2038, 0.1367, 0.1147])
])

# Datasets
train_data = datasets.EuroSAT(root="./data", transform=train_transforms, download=True)
val_data   = datasets.EuroSAT(root="./data", transform=val_transforms,   download=False)

# Split
train_dataset, val_dataset = random_split(train_data, [21600, 5400],
                                          generator=torch.Generator().manual_seed(42))
val_dataset.dataset = val_data

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)

# Training functions
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to('cuda'), labels.to('cuda')
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation"):
            images, labels = images.to('cuda'), labels.to('cuda')
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

print("✅ Everything ready")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"Device: {torch.cuda.get_device_name(0)}")

✅ Everything ready
Train: 21600 | Val: 5400
Device: Tesla T4


In [2]:
model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(512, 10)
model = model.to('cuda')

# See ResNet18 layer structure
for name, param in model.named_parameters():
    print(f"{name:45s} | trainable: {param.requires_grad}")

conv1.weight                                  | trainable: True
bn1.weight                                    | trainable: True
bn1.bias                                      | trainable: True
layer1.0.conv1.weight                         | trainable: True
layer1.0.bn1.weight                           | trainable: True
layer1.0.bn1.bias                             | trainable: True
layer1.0.conv2.weight                         | trainable: True
layer1.0.bn2.weight                           | trainable: True
layer1.0.bn2.bias                             | trainable: True
layer1.1.conv1.weight                         | trainable: True
layer1.1.bn1.weight                           | trainable: True
layer1.1.bn1.bias                             | trainable: True
layer1.1.conv2.weight                         | trainable: True
layer1.1.bn2.weight                           | trainable: True
layer1.1.bn2.bias                             | trainable: True
layer2.0.conv1.weight                   

In [3]:
# Start fresh
model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(512, 10)

# Step 1 — freeze everything
for param in model.parameters():
    param.requires_grad = False

# Step 2 — unfreeze layer4 + fc only
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to('cuda')

# Verify
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,}")

# Use lower LR for fine-tuning — critical
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)  # 10x lower than before

history_ft = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_acc':   []
}

print(f"\n{'Epoch':>5} | {'Train Acc':>9} | {'Val Acc':>8}")
print("-" * 32)

for epoch in range(10):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc     = val_epoch(model, val_loader, criterion)

    history_ft['train_loss'].append(train_loss)
    history_ft['train_acc'].append(train_acc)
    history_ft['val_loss'].append(val_loss)
    history_ft['val_acc'].append(val_acc)

    print(f"  {epoch+1:02d}   |   {train_acc:.4f}  |  {val_acc:.4f}")

print(f"\n=== Results ===")
print(f"Frozen backbone:     0.8530")
print(f"layer4 fine-tuned:   {history_ft['val_acc'][-1]:.4f}")

Trainable: 8,398,858 / 11,181,642

Epoch | Train Acc |  Val Acc
--------------------------------


Validation: 100%|██████████| 169/169 [00:03<00:00, 56.03it/s]


  01   |   0.8161  |  0.9180


Validation: 100%|██████████| 169/169 [00:03<00:00, 49.49it/s]


  02   |   0.8786  |  0.9344


...

  08   |   0.9219  |  0.9417


Validation: 100%|██████████| 169/169 [00:03<00:00, 55.83it/s]


  09   |   0.9284  |  0.9487


Validation: 100%|██████████| 169/169 [00:03<00:00, 45.52it/s]

  10   |   0.9286  |  0.9404

=== Results ===
Frozen backbone:     0.8530
layer4 fine-tuned:   0.9404


In [4]:
# Unfreeze entire network
for param in model.parameters():
    param.requires_grad = True

# Even lower LR — touching early layers is risky
optimizer = optim.Adam(model.parameters(), lr=0.00001)

print(f"{'Epoch':>5} | {'Train Acc':>9} | {'Val Acc':>8}")
print("-" * 32)

for epoch in range(10):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc     = val_epoch(model, val_loader, criterion)

    history_ft['train_loss'].append(train_loss)
    history_ft['train_acc'].append(train_acc)
    history_ft['val_loss'].append(val_loss)
    history_ft['val_acc'].append(val_acc)

    print(f"  {epoch+11:02d}   |   {train_acc:.4f}  |  {val_acc:.4f}")

print(f"\n=== Full Comparison ===")
print(f"Frozen backbone:     0.8530")
print(f"layer4 fine-tuned:   0.9404")
print(f"Full fine-tuned:     {history_ft['val_acc'][-1]:.4f}")

Epoch | Train Acc |  Val Acc
--------------------------------


Validation: 100%|██████████| 169/169 [00:02<00:00, 57.91it/s]


  11   |   0.9400  |  0.9569


Validation: 100%|██████████| 169/169 [00:04<00:00, 42.24it/s]


  12   |   0.9503  |  0.9548


...

  18   |   0.9618  |  0.9602


Validation: 100%|██████████| 169/169 [00:03<00:00, 42.70it/s]


  19   |   0.9629  |  0.9646


Validation: 100%|██████████| 169/169 [00:03<00:00, 55.95it/s]

  20   |   0.9667  |  0.9663

=== Full Comparison ===
Frozen backbone:     0.8530
layer4 fine-tuned:   0.9404
Full fine-tuned:     0.9663


In [5]:
def run_experiment(optimizer_name, lr, epochs=5):
    model = models.resnet18(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(512, 10)
    
    # Unfreeze layer4 + fc — same config as best result
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer4.parameters():
        param.requires_grad = True
    for param in model.fc.parameters():
        param.requires_grad = True
    
    model = model.to('cuda')
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)

    val_loss, val_acc = val_epoch(model, val_loader, criterion)
    print(f"{optimizer_name.upper():>5} | LR: {lr:.0e} | Val Acc: {val_acc:.4f}")
    return val_acc

# Compare
print("=== SGD vs Adam ===")
run_experiment('adam', lr=0.0001)
run_experiment('sgd',  lr=0.001)
run_experiment('sgd',  lr=0.01)

=== SGD vs Adam ===


Validation: 100%|██████████| 169/169 [00:02<00:00, 57.28it/s]


 ADAM | LR: 1e-04 | Val Acc: 0.9470


Validation: 100%|██████████| 169/169 [00:03<00:00, 50.44it/s]


  SGD | LR: 1e-03 | Val Acc: 0.9326


Validation: 100%|██████████| 169/169 [00:02<00:00, 56.84it/s]

  SGD | LR: 1e-02 | Val Acc: 0.8763


0.8762962962962964

In [6]:
model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(512, 10)

for param in model.parameters():
    param.requires_grad = False
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to('cuda')
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Reduce LR by 0.1 every 5 epochs
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

print(f"{'Epoch':>5} | {'LR':>8} | {'Train Acc':>9} | {'Val Acc':>8}")
print("-" * 45)

for epoch in range(10):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc     = val_epoch(model, val_loader, criterion)
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"  {epoch+1:02d}   | {current_lr:.0e} |   {train_acc:.4f}  |  {val_acc:.4f}")
    
    # Step scheduler after each epoch
    scheduler.step()

Epoch |       LR | Train Acc |  Val Acc
---------------------------------------------


Validation: 100%|██████████| 169/169 [00:04<00:00, 40.19it/s]


  01   | 1e-04 |   0.8177  |  0.9220


Validation: 100%|██████████| 169/169 [00:02<00:00, 56.66it/s]


  02   | 1e-04 |   0.8765  |  0.9213


...

  08   | 1e-05 |   0.9296  |  0.9439


Validation: 100%|██████████| 169/169 [00:02<00:00, 56.53it/s]


  09   | 1e-05 |   0.9284  |  0.9448


Validation: 100%|██████████| 169/169 [00:02<00:00, 56.39it/s]

  10   | 1e-05 |   0.9271  |  0.9489
